# Submission 05 - Experiment 13 XGBoost

This submission uses the new best XGBoost configuration from Experiment 13.

Validation ROC-AUC: **0.941776**

The winning change was `subsample=0.90`. All other model settings are kept the same as the previous best configuration.

The model is trained on the full training dataset and used to generate predicted probabilities for `Will_Buy_EV`.


In [1]:
from pathlib import Path
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier

PROJECT_ROOT = Path(r"C:\Users\aakif\Documents\DataCompetition")
TRAIN_PATH = PROJECT_ROOT / "data" / "train.csv"
TEST_PATH = PROJECT_ROOT / "data" / "test.csv"
SAMPLE_PATH = PROJECT_ROOT / "data" / "sample_submission.csv"
OUTPUT_PATH = PROJECT_ROOT / "submissions" / "submission_05.csv"

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
sample_submission = pd.read_csv(SAMPLE_PATH)

X = train.drop(columns=["Will_Buy_EV", "id"])
y = train["Will_Buy_EV"].map({"No": 0, "Yes": 1})
X_test = test.drop(columns=["id"])

numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["number"]).columns.tolist()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

X_processed = preprocessor.fit_transform(X)
X_test_processed = preprocessor.transform(X_test)

print("Training shape:", X_processed.shape)
print("Test shape:", X_test_processed.shape)


Training shape: (668665, 24)
Test shape: (286571, 24)


In [2]:
model = XGBClassifier(
    n_estimators=800,
    max_depth=5,
    learning_rate=0.04,
    min_child_weight=2,
    subsample=0.90,
    colsample_bytree=0.85,
    gamma=0,
    reg_alpha=0,
    reg_lambda=1,
    objective="binary:logistic",
    eval_metric="auc",
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

model.fit(X_processed, y)

test_predictions = model.predict_proba(X_test_processed)[:, 1]

submission = pd.DataFrame({
    "id": test["id"],
    "Will_Buy_EV": test_predictions
})

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
submission.to_csv(OUTPUT_PATH, index=False)

print(f"Created: {OUTPUT_PATH}")
print("Submission shape:", submission.shape)
print(submission.head())


Created: C:\Users\aakif\Documents\DataCompetition\submissions\submission_05.csv
Submission shape: (286571, 2)
       id  Will_Buy_EV
0  668665     0.010253
1  668666     0.015790
2  668667     0.005057
3  668668     0.003479
4  668669     0.021052


In [3]:
assert submission.shape == sample_submission.shape
assert list(submission.columns) == list(sample_submission.columns)
assert submission["id"].equals(sample_submission["id"])
assert submission["Will_Buy_EV"].notna().all()
assert submission["Will_Buy_EV"].between(0, 1).all()

print("=" * 60)
print("SUBMISSION 05 VERIFIED")
print("=" * 60)
                
print(f"Validation ROC-AUC: 0.941776")
print("Model: XGBoost")
print("subsample: 0.90")
print("All submission checks passed.")


SUBMISSION 05 VERIFIED
Validation ROC-AUC: 0.941776
Model: XGBoost
subsample: 0.90
All submission checks passed.
